# ALE 3D CNN NeuroVLM Experiment

Runs both ALE3DCNN variants explicitly: DiFuMo-compatible and atlas-free. Outputs are written to Google Drive, while packed caches are copied to local Colab disk for faster training.

## 1. Runtime / GPU Check

In [ ]:
import os, platform, subprocess, sys, time, shutil, json
print('Python:', sys.version)
try:
    import torch
    print('Torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('Torch check failed:', exc)
print('Platform:', platform.platform())

## 2. Dependency Install

In [ ]:
# PyG is not required for the dense ALE CNN path.
!pip -q install nilearn nibabel huggingface-hub safetensors adapters transformers pyarrow matplotlib pandas scikit-learn tqdm umap-learn

## 3. Repo Clone / Pull

In [ ]:
REPO_URL = 'https://github.com/neurovlm/neurovlm.git'
REPO_BRANCH = 'neurovlm_gnn'
REPO_DIR = '/content/neurovlm_gnn'
if not os.path.exists(REPO_DIR):
    !git clone --branch $REPO_BRANCH --single-branch $REPO_URL $REPO_DIR
else:
    !git -C $REPO_DIR fetch origin $REPO_BRANCH
    !git -C $REPO_DIR checkout $REPO_BRANCH
    !git -C $REPO_DIR pull --ff-only origin $REPO_BRANCH
os.chdir(REPO_DIR)
!pip -q install -e '.[viz,notebook,metrics]'
print('Working directory:', os.getcwd())

## 4. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 5. Shared Config

In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/neurovlm_ale3dcnn'
DRIVE_CACHE_DIR = f'{DRIVE_ROOT}/data/ale_caches'
LOCAL_CACHE_DIR = '/content/ale_caches'
RUN_NAME_OR_TIMESTAMP = time.strftime('%Y%m%d_%H%M%S')

EPOCHS = 200
BATCH_SIZE = 16
RESOLUTION_MM = 4
KERNEL_FWHM_MM = 9
BASE_CHANNELS = 16
NUM_BLOCKS = 3
OUT_DIM = 384
DROPOUT = 0.1
VAL_INTERVAL = 5
EARLY_STOPPING_PATIENCE = 25
MONITOR_METRIC = 'paper_recall_curve_auc'
CACHE_DTYPE = 'float16'
NUM_WORKERS = 2
COMPARISON_FILE = f'{DRIVE_ROOT}/runs/ale_model_comparison.csv'

os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
os.makedirs(LOCAL_CACHE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/runs', exist_ok=True)

SHARED_CONFIG = {
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'resolution_mm': RESOLUTION_MM,
    'kernel_fwhm_mm': KERNEL_FWHM_MM,
    'base_channels': BASE_CHANNELS,
    'num_blocks': NUM_BLOCKS,
    'out_dim': OUT_DIM,
    'dropout': DROPOUT,
    'val_interval': VAL_INTERVAL,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'monitor_metric': MONITOR_METRIC,
    'cache_dtype': CACHE_DTYPE,
}
with open(f'{DRIVE_ROOT}/shared_training_config.json', 'w') as f:
    json.dump(SHARED_CONFIG, f, indent=2)
print(json.dumps(SHARED_CONFIG, indent=2))

## 6. Helper Function

In [ ]:
COMPLETED_RUNS = []

def _cache_name(mode):
    return f'{mode}_{RESOLUTION_MM}mm_fwhm{KERNEL_FWHM_MM}_crop_{CACHE_DTYPE}.pt'

def run_ale_experiment(mode, model, run_name):
    drive_run_dir = f'{DRIVE_ROOT}/runs/{model}_{mode}_{run_name}'
    local_cache_file = f'{LOCAL_CACHE_DIR}/{_cache_name(mode)}'
    drive_cache_file = f'{DRIVE_CACHE_DIR}/{_cache_name(mode)}'
    os.makedirs(drive_run_dir, exist_ok=True)
    os.makedirs(LOCAL_CACHE_DIR, exist_ok=True)
    os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)

    local_cache_existed = os.path.exists(local_cache_file)
    if os.path.exists(drive_cache_file) and not os.path.exists(local_cache_file):
        print(f'Copying Drive cache to local disk: {drive_cache_file} -> {local_cache_file}')
        shutil.copy2(drive_cache_file, local_cache_file)
    elif os.path.exists(local_cache_file):
        print(f'Using existing local cache: {local_cache_file}')
    else:
        print(f'No cache found; training script will build local packed cache: {local_cache_file}')

    cmd = [
        sys.executable, 'experiments/train_ale_cnn.py',
        '--mode', mode,
        '--model', model,
        '--epochs', str(EPOCHS),
        '--batch-size', str(BATCH_SIZE),
        '--cache-file', local_cache_file,
        '--run-dir', drive_run_dir,
        '--checkpoint-dir', f'{drive_run_dir}/checkpoints',
        '--comparison-file', COMPARISON_FILE,
        '--resolution-mm', str(RESOLUTION_MM),
        '--kernel-fwhm-mm', str(KERNEL_FWHM_MM),
        '--base-channels', str(BASE_CHANNELS),
        '--num-blocks', str(NUM_BLOCKS),
        '--out-dim', str(OUT_DIM),
        '--dropout', str(DROPOUT),
        '--cache-dtype', CACHE_DTYPE,
        '--val-interval', str(VAL_INTERVAL),
        '--early-stopping-patience', str(EARLY_STOPPING_PATIENCE),
        '--monitor-metric', MONITOR_METRIC,
        '--num-workers', str(NUM_WORKERS),
        '--amp',
        '--umap',
    ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)

    if os.path.exists(local_cache_file) and (not os.path.exists(drive_cache_file) or not local_cache_existed):
        print(f'Copying local packed cache back to Drive: {local_cache_file} -> {drive_cache_file}')
        shutil.copy2(local_cache_file, drive_cache_file)

    run_info = {
        'mode': mode,
        'model': model,
        'drive_run_dir': drive_run_dir,
        'local_cache_file': local_cache_file,
        'drive_cache_file': drive_cache_file,
        'best_checkpoint': f'{drive_run_dir}/checkpoints/best_ale_cnn.pt',
        'last_checkpoint': f'{drive_run_dir}/checkpoints/last_ale_cnn.pt',
        'eval_results': f'{drive_run_dir}/eval_results.json',
        'training_history': f'{drive_run_dir}/training_history.json',
        'comparison_row': f'{drive_run_dir}/comparison_row.json',
    }
    COMPLETED_RUNS.append(run_info)
    with open(f'{drive_run_dir}/colab_run_info.json', 'w') as f:
        json.dump(run_info, f, indent=2)
    print(json.dumps(run_info, indent=2))
    return run_info

## 7. Cell 1 - Run DiFuMo-Compatible ALE3DCNN

In [ ]:
difumo_run = run_ale_experiment(
    mode='difumo_compatible',
    model='ale_3dcnn',
    run_name=RUN_NAME_OR_TIMESTAMP,
)

## 8. Cell 2 - Run Atlas-Free ALE3DCNN

In [ ]:
atlas_free_run = run_ale_experiment(
    mode='atlas_free',
    model='ale_3dcnn',
    run_name=RUN_NAME_OR_TIMESTAMP,
)

## 9. Results Saved

In [ ]:
import pandas as pd
rows = COMPLETED_RUNS
if not rows:
    rows = [x for x in [globals().get('difumo_run'), globals().get('atlas_free_run')] if x]
pd.DataFrame(rows)

## 10. Comparison Notebook

The comparison CSV is saved at `DRIVE_ROOT/runs/ale_model_comparison.csv`. Run `experiments/compare_ale_models.ipynb` after both cells finish to make side-by-side tables and plots. If you want the comparison notebook to read Drive automatically, set `COMPARISON_FILE` there to the same path printed above.